# Unconstrained bi-objective optimization - Bi-SEGO


In [ ]:
from smt_optim.acquisition_strategies.bisego import BiSEGO

import numpy as np

from smt_optim.core import Problem
from smt_optim.surrogate_models.smt import SmtAutoModel
from smt_optim.core import ObjectiveConfig, DriverConfig
from smt_optim.core import Driver

from smt_optim.benchmarks.registry import list_problems, get_problem

import matplotlib.pyplot as plt

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.hv import HV
from pymoo.core.evaluator import Evaluator
from pymoo.core.population import Population

from smt_optim.utils.inject_doe import InjectData
from smt.sampling_methods import LHS
import smt.design_space as ds

from datetime import datetime
import pickle

### Parameters for optimization

In [ ]:
n_accuracy=100 # Precision on composite acquisition function
seed=421 # Seed for generating the initial DoE
budget_factor=20 # Budget: budget_factor * dim
init_factor=2 # Initial DoE size: init_factor * dim + 1
min_factor=2 # Initial calls to determine min(f1): min_factor * dim + 1 (same for min(f2))
max_so_iter_factor=2 # Max calls to a single-objective subproblem: max_so_iter_factor * dim + 1
soformulation_naive="Normalized"
soformulation_composite="Product"
multi_start_factor=10 # Number of multistart calls for acquisition function optimization: multi_start_factor * dim
test_number=0
surrogate=SmtAutoModel

parameters={
    "n_accuracy":n_accuracy,
    "seed":seed,
    "budget_factor":budget_factor,
    "init_factor":init_factor,
    "min_factor":min_factor,
    "max_so_iter_factor":max_so_iter_factor,
    "soformulation_composite":soformulation_composite,
    "soformulation_naive":soformulation_naive,
    "multi_start_factor":multi_start_factor,
    "test_number":test_number,
    "surrogate":surrogate,
}

### Importing a test function

Here, we are using the analytical test function ZDT1 in 2 dimensions as an example.

In [ ]:
bproblem = get_problem("ZDT1_D02")

### Initialize the problem:

In [ ]:
name=bproblem.name
num_dim=bproblem.num_dim
num_obj=bproblem.num_obj
num_cstr=bproblem.num_cstr
bounds=bproblem.bounds
objective=bproblem.objective

assert(num_obj==2)
assert(num_cstr==0)

f1=objective[0]
f2=objective[1]

max_budget=budget_factor*num_dim
n_init=init_factor*num_dim+1
n_min=min_factor*num_dim+1
n_so=max_so_iter_factor*num_dim+1

obj_config1 = ObjectiveConfig(
    [f1],
    type="minimize",
    surrogate=surrogate,
)

obj_config2 = ObjectiveConfig(
    [f2],
    type="minimize",
    surrogate=surrogate,
)

prob_definition = Problem(
    obj_configs=[obj_config1,obj_config2],
    design_space=bounds,
    costs=[1,1]
)

Initialize the DoE using Latin Hypercube Sampling

In [ ]:
float_vars = []
for idx in range(bounds.shape[0]):
    float_vars.append(
        ds.FloatVariable(bounds[idx, 0], bounds[idx, 1])
    )
design_space = ds.DesignSpace(float_vars)

F=lambda x: (f1(x),f2(x))
sampler = LHS(xlimits=design_space.get_unfolded_num_bounds(),
                            criterion="ese",
                            seed=seed, )
doe = sampler(n_init)
D=[x for x in doe]
Y=[F(x) for x in D]

Initialize the driver and run the optimization

In [ ]:
opt_config = DriverConfig(
    max_iter = max_budget - n_init,
    max_budget = max_budget,
    nt_init = n_init+2*n_min,
    verbose = True,
    scaling = True,
    seed=seed,
)

strategy_kwargs = {
    "n_multi_start":multi_start_factor*num_dim,
    "n_init":n_init,
    "n_accuracy":n_accuracy,
    "so_formulation":soformulation_composite,
    "single_objective_max_calls":n_so,
    "min_max_calls":n_min,
    "naive":False,
}

xt_init = np.vstack([np.atleast_1d(x) for x in D])
yt_init = np.vstack([np.atleast_1d(y) for y in Y])

driver = Driver(prob_definition, opt_config, strategy=BiSEGO, strategy_kwargs=strategy_kwargs)

InjectData(driver.state, xt_init, yt_init)

state = driver.optimize()

In [ ]:
def get_DoE(state):
    # Gets the DoE in the format (D,Y) where D is the list of evaluation points and Y is the list of objective function values
    Y=state.dataset.export_as_dict()["obj"]
    D=state.dataset.export_as_dict()["x"]
    return (D,Y)

def dominates(p,q):
    # Returns True if point p strictly dominates point q, else returns False
    return (p[0]<q[0] and p[1]<=q[1]) or (p[0]<=q[0] and p[1]<q[1])

def get_Pareto_front(D,Y):
    # Given a DoE (D,Y), returns the list of indices of non-dominated points, sorted by ascending value of f1
    t=len(D)
    front=[]
    for i in range(t):
        if all([not dominates(q,Y[i]) for q in Y]):
            front.append((Y[i][0],i))
    front.sort()
    return [p[1] for p in front]

Obtain the true Pareto front using pymoo with high budget

In [ ]:
class PymooProblem(ElementwiseProblem):
    # Formulates a given benchmark problem as a problem solvable by pymoo

    def __init__(self,bprob):
        self.prob=bprob
        super().__init__(n_var=self.prob.num_dim,
                         n_obj=self.prob.num_obj,
                         xl=np.array([coord_bounds[0] for coord_bounds in self.prob.bounds]),
                         xu=np.array([coord_bounds[1] for coord_bounds in self.prob.bounds]))

    def _evaluate(self, x, out, *args, **kwargs):
        f1 = self.prob.objective[0](x)
        f2 = self.prob.objective[1](x)

        out["F"] = [f1, f2]

def run_benchmark_pymoo(bproblem):
    # Runs pymoo with high budget on a given benchmark problem. Returns a list of the Pareto optimal points and a list of the corresponding points in the objective space

    pymoo_problem=PymooProblem(bproblem)
    algorithm = NSGA2(
        pop_size=1000,
        n_offsprings=250,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=15),
        mutation=PM(eta=20),
        eliminate_duplicates=True
    )
    termination = get_termination("n_gen", 1000)
    res = minimize(pymoo_problem,
                algorithm,
                termination,
                seed=1,
                save_history=True,
                verbose=True)

    X = res.X
    F = res.F
    return (X,F)

X_true,F_true=run_benchmark_pymoo(bproblem)

In [ ]:
fig,ax=plt.subplots(1,1)

D,Y=get_DoE(state)


pareto_points = [(D[i],Y[i]) for i in get_Pareto_front(D,Y)]
ax.scatter([p[1][0] for p in pareto_points],[p[1][1] for p in pareto_points],color="blue",label="Composite Bi-SEGO")

pareto_points_pymoo = [(X_true[i],F_true[i]) for i in range(len(X_true))]
ax.scatter([p[1][0] for p in pareto_points_pymoo],[p[1][1] for p in pareto_points_pymoo],marker=".",s=5,color="red",label="Optimal Pareto front")

ax.legend(loc="best")
ax.set_xlabel("f1")
ax.set_ylabel("f2")
ax.title.set_text("Pareto Front obtained for the problem ZDT1 for $d = 2$")
ax.legend(loc="best")
ax.set_xlabel("f1")
ax.set_ylabel("f2")

plt.show()